# HG4052 · Week 6 Practical
## Decode it by hand, then in code

**No installs: everything runs on numpy and matplotlib, preinstalled on Colab. No audio this week: the sound has already been turned into per-frame scores for you.**

By the end you will have:
- ✅ the ice-cream trellis from the paper sheet, checked against the machine's
- ✅ Viterbi in logs, with the one line that is the whole idea written by you
- ✅ a decoder that tells "no" from "go" on ten clips, and plots where /n/ or /g/ ends and /oʊ/ begins: a forced alignment
- ✅ five candidate transcripts of one Singlish sentence reranked by two priors, a news one and a Singapore one, and the winner changing
- ✅ a decoder that cannot print "makan" because its lexicon lacks the word, and your two-line diagnosis
- ✅ (take-home, pick one) add-one smoothing with held-out perplexity, ten sampled sentences, or a third word in the decoder

**How this notebook works.** Same as every week: click a cell, press **Shift + Enter**, and read the output underneath. Cells marked **✏️ TODO** have one small blank to fill (always one line or less). Every function is provided for you to read and run, never to write. Written TODOs are answered by double-clicking the cell and typing.

**Short on time?** Prioritise **Setup → Part 2 → Part 3 → Part 4**: the recurrence, the decoder and the rerank are the week. Part 1 is a five-minute check of the paper sheet, Part 5 is ten minutes, Part 6 is take-home by design.

---
### Before this notebook: the paper block

Ten minutes with a pencil and the printed sheet (the ice-cream diary from Appendix A of the textbook). Two states, Hot and Cold; three days; the diary reads 3 1 3.

- Day 1: each cell = start probability × emission of that day's number.
- Every later cell: emission of that day's number × the larger of the two (previous cell × transition) products; draw an arrow from the winner.
- Day 3: take the larger cell and follow the arrows back. That is the weather sequence, and its probability.
- Then relabel: H = oral vowel, C = nasal vowel; 1, 2, 3 = nasal murmur levels. Nothing in the arithmetic changes.

Keep the sheet; Part 1 prints the machine's trellis and you compare.

---
## 0 · Setup

Week 1's command first: where are you standing?

In [ ]:
!pwd

**Imports.** Nothing new: `numpy`, `matplotlib`, and the `csv` module to read today's tables. Today's new idea is one line of arithmetic, and you write it yourself in Part 2.

In [ ]:
import os, csv, math, urllib.request
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
print("numpy", np.__version__, "loaded: the toolbox is open")

**Get today's data.** Five small text files from the course repo (a few kilobytes; no audio). All of them are **synthetic**, made for this notebook: the per-frame scores were not measured from real recordings, and the bigram counts are invented, not taken from a real corpus. The cell reports what it found.

| file | what it holds |
|---|---|
| `state_loglik_no_go.csv` | for ten clips, one row per frame: the log-likelihood of that frame under three states, /n/, /g/ and /oʊ/ |
| `clips_truth.csv` | which word each clip is, and where its onset ends (for checking, not for decoding) |
| `bigram_counts.csv` | two bigram-count tables over one small vocabulary: `news` and `nsc` |
| `candidates.csv` | five candidate transcripts of one utterance, each with its acoustic score |
| `heldout_lines.txt` | four Singlish lines the tables never saw (for the stretch) |

In [ ]:
FILES = ["state_loglik_no_go.csv", "clips_truth.csv", "bigram_counts.csv", "candidates.csv", "heldout_lines.txt"]
os.makedirs("data", exist_ok=True)
for name in FILES:
    path = os.path.join("data", name)
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week06/" + name, path)
        except Exception as e:
            print("download failed for", name, ":", e)
    ok = os.path.exists(path) and os.path.getsize(path) > 50
    print(("✅" if ok else "❌"), name, (f"{os.path.getsize(path):,} bytes" if ok else "missing (also on NTULearn: download and upload into data/)"))

---
## 1 · The ice-cream trellis, checked

The function below is finished: it runs Viterbi on the ice-cream HMM in plain probabilities, exactly as you did on paper, and prints the trellis with an arrow into each cell from its winning predecessor. Read the `for` loops: two nested, one over days and one over states, and inside them one `max`.

In [ ]:
ICE = {
    "states": ["H", "C"],
    "start":  {"H": 0.8, "C": 0.2},
    "trans":  {("H", "H"): 0.6, ("H", "C"): 0.4, ("C", "H"): 0.5, ("C", "C"): 0.5},
    "emis":   {"H": {1: 0.2, 2: 0.4, 3: 0.4}, "C": {1: 0.5, 2: 0.4, 3: 0.1}},
}
DIARY = [3, 1, 3]

def viterbi_prob(hmm, obs):
    # Viterbi in plain probabilities; returns the table v[t][state], the winners, and the best path.
    S = hmm["states"]
    v, who = [], []
    v.append({s: hmm["start"][s] * hmm["emis"][s][obs[0]] for s in S}); who.append({s: None for s in S})
    for t in range(1, len(obs)):
        row, arrows = {}, {}
        for s in S:
            cand = {p: v[t - 1][p] * hmm["trans"][(p, s)] for p in S}   # (previous cell × transition), one per previous state
            best = max(cand, key=cand.get)                                # the winner
            row[s] = hmm["emis"][s][obs[t]] * cand[best]                 # emission × the largest product
            arrows[s] = best
        v.append(row); who.append(arrows)
    last = max(v[-1], key=v[-1].get)
    path = [last]
    for t in range(len(obs) - 1, 0, -1):                                 # follow the arrows back
        path.append(who[t][path[-1]])
    return v, who, path[::-1]

v, who, path = viterbi_prob(ICE, DIARY)
print("        " + "   ".join(f"day {t+1} (ate {o})" for t, o in enumerate(DIARY)))
for s in ICE["states"]:
    cells_ = [f"{v[0][s]:.4f}"] + [f"{v[t][s]:.4f} ←{who[t][s]}" for t in range(1, len(DIARY))]
    print(f"  {s}    " + "     ".join(f"{c:<14}" for c in cells_))
print(f"\nbest path {' '.join(path)}, probability {v[-1][path[-1]]:.4f}")
print("runner-up H H H:", round(0.8*0.4 * 0.6*0.2 * 0.6*0.4, 4))

**✏️ TODO (in words, double-click).** Does the machine's trellis match your sheet, cell by cell and arrow by arrow? Which day's two cells were closest, and what in the diary made that day hard to call? After relabelling (oral/nasal vowel, murmur levels), what changed?

> **...**

---
## 2 · Viterbi in logs: your line

Real decoders multiply hundreds of probabilities, so they work in logs: every × becomes +, and the max is unchanged (the largest product has the largest log). The recurrence from the lecture, in logs:

`V[t, s] = log P(x_t | s) + max over prev of ( V[t−1, prev] + log P(prev → s) )`

Below, the model is three arrays: `log_start` (one per state), `log_trans` (rows = from, columns = to) and `log_emis` (rows = frames, columns = states). The two `for` loops are written; the line inside them is yours. Its building blocks are on the comment line above it. If the line stays unfinished, a finished copy (`viterbi_reference`) takes over further down so the rest of the notebook still runs.

In [ ]:
def viterbi(log_start, log_trans, log_emis):
    # Returns the table V (frames × states), the backpointers B, and the best state path (one state index per frame).
    T, S = log_emis.shape
    V = np.full((T, S), -np.inf)
    B = np.zeros((T, S), dtype=int)
    V[0] = log_start + log_emis[0]                                   # frame 1: start + emission (no previous cell)
    for t in range(1, T):
        for s in range(S):
            cand = V[t - 1] + log_trans[:, s]                        # (previous cell + transition), one per previous state
            # ✏️ TODO: this cell = this frame's emission under s + the largest candidate.
            # Building blocks:   log_emis[t, s]     np.max(cand)
            V[t, s] = ...
            B[t, s] = np.argmax(cand)                                # the backpointer: which previous state won
    return V, B, backtrace(V, B)

def backtrace(V, B):
    # Start at the best final cell and follow the backpointers to frame 1.
    T = V.shape[0]
    path = [int(np.argmax(V[-1]))]
    for t in range(T - 1, 0, -1):
        path.append(int(B[t, path[-1]]))
    return path[::-1]

def viterbi_reference(log_start, log_trans, log_emis):
    # The finished recurrence, for the fallback below (and to read if you are stuck: it IS the answer).
    T, S = log_emis.shape
    V = np.full((T, S), -np.inf); B = np.zeros((T, S), dtype=int)
    V[0] = log_start + log_emis[0]
    for t in range(1, T):
        for s in range(S):
            cand = V[t - 1] + log_trans[:, s]
            V[t, s] = log_emis[t, s] + np.max(cand)
            B[t, s] = np.argmax(cand)
    return V, B, backtrace(V, B)

# the ice-cream HMM as arrays, in logs (state order H, C; observations 1, 2, 3)
log_start_ice = np.log([0.8, 0.2])
log_trans_ice = np.log([[0.6, 0.4],
                        [0.5, 0.5]])
emis_ice = np.array([[0.2, 0.4, 0.4],       # P(1|H) P(2|H) P(3|H)
                     [0.5, 0.4, 0.1]])      # P(1|C) P(2|C) P(3|C)
log_emis_diary = np.log(emis_ice[:, [o - 1 for o in DIARY]].T)   # frames × states

try:
    V_ice, B_ice, path_ice = viterbi(log_start_ice, log_trans_ice, log_emis_diary)
    TODO_DONE = bool(np.all(np.isfinite(V_ice)))
except TypeError:
    TODO_DONE = False
VIT = viterbi if TODO_DONE else viterbi_reference
if not TODO_DONE:
    print("⬆ TODO not finished: using viterbi_reference for now (fill the line and re-run this cell).")
    V_ice, B_ice, path_ice = VIT(log_start_ice, log_trans_ice, log_emis_diary)

names = np.array(ICE["states"])
print("V in logs:\n", V_ice.T)
print("V back in probabilities:\n", np.exp(V_ice).T)
print("best path:", " ".join(names[path_ice]), " probability", round(float(np.exp(V_ice[-1].max())), 4))
print("✅ matches Part 1" if list(names[path_ice]) == path and abs(np.exp(V_ice[-1].max()) - v[-1][path[-1]]) < 1e-9 else "❌ does not match Part 1: check the line")

**Why logs.** The diary was three days; a recording is a hundred frames a second. Multiply a few hundred probabilities and the product falls below the smallest number a float can hold, and becomes exactly 0.0, with no error. Logs add instead, and the sum just becomes a large negative number.

In [ ]:
rng = np.random.default_rng(6)
p = rng.choice([0.2, 0.4, 0.5], size=1000)          # one thousand emission probabilities: a ten-second recording
print("product of 1,000 probabilities :", np.prod(p))
print("sum of their logs              :", round(float(np.sum(np.log(p))), 1))
print("Week 2's line, for comparison  :", 0.01 ** 200)

---
## 3 · Decode "no" vs "go"

Two word HMMs share their vowel: **no** = /n/ → /oʊ/ and **go** = /g/ → /oʊ/. Each state has a self-loop (0.8) and a forward arrow (0.2), and a word must start in its first state and end in its last. The acoustic model's work has been done for you: `state_loglik_no_go.csv` gives, for every frame of ten clips, the log-likelihood of that frame under /n/, /g/ and /oʊ/ (**synthetic** numbers on a realistic scale, about −1.5 for a frame that fits a state and −5 for one that does not).

The decoder is the lecture's: run Viterbi through each word's chain, compare the two final scores, print the word with the larger. The backtraced path also says where the onset ends and the vowel begins: a segmentation nobody marked by hand, which is what a forced aligner produces.

In [ ]:
def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

LL = {}                                                     # clip -> (frames × 3) array of log-likelihoods, columns n, g, ou
for r in read_csv("data/state_loglik_no_go.csv"):
    LL.setdefault(r["clip"], []).append([float(r["ll_n"]), float(r["ll_g"]), float(r["ll_ou"])])
LL = {c: np.array(rows) for c, rows in LL.items()}
TRUTH = {r["clip"]: r for r in read_csv("data/clips_truth.csv")}
COL = {"n": 0, "g": 1, "ou": 2}

WORDS = {"no": ["n", "ou"], "go": ["g", "ou"]}
STAY, MOVE = 0.8, 0.2

def word_model(states):
    S = len(states)
    log_start = np.full(S, -np.inf); log_start[0] = 0.0                       # the word begins in its first state
    log_trans = np.full((S, S), -np.inf)
    for i in range(S):
        log_trans[i, i] = np.log(STAY)
        if i + 1 < S:
            log_trans[i, i + 1] = np.log(MOVE)
    return log_start, log_trans

def decode(clip, words=WORDS):
    scores, paths = {}, {}
    for w, states in words.items():
        log_start, log_trans = word_model(states)
        log_emis = LL[clip][:, [COL[s] for s in states]]                         # frames × states of this word
        V, B, path = VIT(log_start, log_trans, log_emis)
        scores[w] = float(V[-1, -1])                                             # the word must end in its last state
        paths[w] = path
    best = max(scores, key=scores.get)
    return best, scores, paths

print(f"{'clip':7} {'truth':6} {'decoded':8} {'score no':>9} {'score go':>9}  boundary (first vowel frame, truth)")
correct = 0
for clip in sorted(LL):
    best, scores, paths = decode(clip)
    boundary = paths[best].index(1) if 1 in paths[best] else None
    correct += best == TRUTH[clip]["word"]
    flag = "" if best == TRUTH[clip]["word"] else "  ← confusion"
    print(f"{clip:7} {TRUTH[clip]['word']:6} {best:8} {scores['no']:9.1f} {scores['go']:9.1f}  {boundary} ({TRUTH[clip]['onset_frames']}){flag}")
print(f"\naccuracy: {correct}/10")

In [ ]:
CLIP = "clip05"                                  # change and re-run; try a confused clip
best, scores, paths = decode(CLIP)
ll = LL[CLIP]
fig, axes = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
axes[0].plot(ll[:, 0], label="/n/", color="#0e6f66"); axes[0].plot(ll[:, 1], label="/g/", color="#d9472b"); axes[0].plot(ll[:, 2], label="/oʊ/", color="#b07c14")
axes[0].set_ylabel("log-likelihood of the frame"); axes[0].legend(loc="lower right"); axes[0].set_title(f"{CLIP}: truth {TRUTH[CLIP]['word']}, decoded {best}  (no {scores['no']:.1f}, go {scores['go']:.1f})")
for w, col in (("no", "#0e6f66"), ("go", "#d9472b")):
    axes[1].step(range(len(paths[w])), paths[w], where="mid", color=col, lw=2, label=f"path under '{w}'")
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["onset state", "/oʊ/"]); axes[1].set_xlabel("frame"); axes[1].legend(loc="lower right")
b = paths[best].index(1) if 1 in paths[best] else None
if b is not None:
    for ax in axes: ax.axvline(b - 0.5, color="#333", ls="--", lw=1)
    axes[1].text(b, 0.5, f"  boundary after frame {b}", va="center")
plt.tight_layout(); plt.show()
print("The dashed line is the decoded boundary: the frame where the best path leaves the onset state for /oʊ/.")
print("Nobody marked it; the model inferred it. That is a forced alignment, the tool Week 9 uses on real recordings.")

**✏️ TODO (in words, double-click).** Take a confused clip (the table marks them; set `CLIP` to it and re-run the plot). Which frames scored /n/ above /g/, or the reverse, and where are they in the onset? Why does one wrong frame not flip the decision on the clear clips, and what was different about this one? Then: is the boundary found by "no" in the same place as the boundary found by "go"?

> **...**

---
## 4 · The prior at work: rerank five candidates

One Singlish utterance, five candidate transcripts, each with an **acoustic score** (log10, provided; the acoustics slightly prefer "mark can" over "makan", which sound alike). Two bigram tables give the **prior**: `news`, counted from newswire-flavoured text, and `nsc`, counted from Singapore-conversation-flavoured text (both invented for this notebook; the real National Speech Corpus is the model for the second). The decoder's rule is the lecture's equation in logs:

`total = acoustic score + λ × prior score`, and the largest total wins.

Unseen bigrams get a floor of 10⁻⁶ instead of zero, so that no candidate is exactly impossible (Part 6 does this properly with smoothing).

In [ ]:
COUNTS = {"news": {}, "nsc": {}}
for r in read_csv("data/bigram_counts.csv"):
    COUNTS[r["corpus"]][(r["prev"], r["next"])] = int(r["count"])
PREV_TOTAL = {c: {} for c in COUNTS}
for c, table in COUNTS.items():
    for (a, b), n in table.items():
        PREV_TOTAL[c][a] = PREV_TOTAL[c].get(a, 0) + n

def lm_score(sentence, corpus, floor=1e-6):
    # log10 P(sentence) under a bigram table, with sentence boundaries; unseen pairs get the floor.
    words = ["<s>"] + sentence.split() + ["</s>"]
    total = 0.0
    for a, b in zip(words, words[1:]):
        n_ab, n_a = COUNTS[corpus].get((a, b), 0), PREV_TOTAL[corpus].get(a, 0)
        p = n_ab / n_a if n_ab else floor
        total += math.log10(p)
    return total

CANDS = [(r["candidate"], float(r["am_log10"])) for r in read_csv("data/candidates.csv")]
print(f"{'candidate':26} {'AM':>6} {'LM news':>8} {'LM nsc':>7} {'total news':>11} {'total nsc':>10}")
for sent, am in CANDS:
    ln, ls = lm_score(sent, "news"), lm_score(sent, "nsc")
    print(f"{sent:26} {am:6.1f} {ln:8.1f} {ls:7.1f} {am+ln:11.1f} {am+ls:10.1f}")
for corpus in ("news", "nsc"):
    win = max(CANDS, key=lambda c: c[1] + lm_score(c[0], corpus))
    print(f"winner with the {corpus} prior (λ = 1): {win[0]!r}")
print("acoustics alone (λ = 0):", max(CANDS, key=lambda c: c[1])[0])

In [ ]:
# the LM weight: how much the prior is allowed to say
for lam in (0, 0.25, 0.5, 1, 2, 4, 8):
    win = max(CANDS, key=lambda c: c[1] + lam * lm_score(c[0], "nsc"))
    print(f"λ = {lam:<5} with the nsc prior → {win[0]!r}")

**✏️ TODO (in words, double-click).** From the sweep: the value of λ at which the prior first rescues the right transcript, and the value at which it starts overriding the acoustics (which word disappears, and why does a shorter sentence get a higher prior?). Then one sentence each on a real situation where each failure harms a speaker: a prior too weak, and a prior too strong.

> **...**

---
## 5 · Break it: the missing word

The third model in the classical decoder is the **lexicon**: the pronouncing dictionary that maps every word to its phone states. A word the lexicon lacks cannot be printed, whatever the audio says. Below, the decoder is the Part 4 reranker with a lexicon check: any candidate containing a word outside the lexicon is impossible.

In [ ]:
LEXICON = {   # word -> phones (ARPAbet-style); "makan" is not in it
    "later": "L EY T ER", "late": "L EY T", "we": "W IY", "go": "G OW", "lah": "L AA", "mark": "M AA R K",
    "can": "K AE N", "my": "M AY", "car": "K AA R", "market": "M AA R K IH T", "home": "HH OW M",
    "the": "DH AH", "i": "AY", "eat": "IY T", "and": "AE N D",
}

def decode_with_lexicon(cands, corpus="nsc", lam=1.0, lexicon=LEXICON):
    scored = []
    for sent, am in cands:
        missing = [w for w in sent.split() if w not in lexicon]
        total = -np.inf if missing else am + lam * lm_score(sent, corpus)
        scored.append((total, sent, missing))
    scored.sort(reverse=True)
    return scored

for total, sent, missing in decode_with_lexicon(CANDS):
    print(f"{total:8.1f}  {sent:26} {'impossible: ' + ', '.join(missing) + ' not in the lexicon' if missing else ''}")
print("\nThe decoder prints:", decode_with_lexicon(CANDS)[0][1])

LEXICON2 = dict(LEXICON, makan="M AA K AH N")
print("With makan added :", decode_with_lexicon(CANDS, lexicon=LEXICON2)[0][1])

**✏️ TODO (two lines, double-click).** What did the decoder print for a speaker who said "makan", and why could no amount of acoustic evidence fix it? Name the two changes that would (one to the lexicon, one to the prior).

> **...**

---
## 6 · Stretch (pick one, take-home)

**(a) Add-one smoothing and held-out perplexity.** Replace the floor with Laplace smoothing, count(prev, next) + 1 over count(prev) + V, and score four Singlish lines the tables never saw. Perplexity = 10^(−(1/N) × Σ log10 P), N = number of predicted words. Which prior is less surprised by Singlish, and by how much?

In [ ]:
VOCAB = sorted({w for c in COUNTS for pair in COUNTS[c] for w in pair})
V_SIZE = len(VOCAB)

def lm_score_add1(sentence, corpus):
    words = ["<s>"] + sentence.split() + ["</s>"]
    return sum(math.log10((COUNTS[corpus].get((a, b), 0) + 1) / (PREV_TOTAL[corpus].get(a, 0) + V_SIZE)) for a, b in zip(words, words[1:]))

HELDOUT = [l.strip() for l in open("data/heldout_lines.txt") if l.strip()]
for corpus in ("news", "nsc"):
    logp = sum(lm_score_add1(l, corpus) for l in HELDOUT)
    n_pred = sum(len(l.split()) + 1 for l in HELDOUT)            # each word plus the end token
    print(f"{corpus}: perplexity on the held-out lines = {10 ** (-logp / n_pred):.1f}")

**(b) Sample ten sentences from each table.** Week 2's babbling machine, over words: start at `<s>`, draw the next word with the row's counts as weights, stop at `</s>`. Judge them: which table's sentences sound like something a person in Singapore would say, and where do both tables go wrong?

In [ ]:
import random
random.seed(6)
def sample(corpus, max_len=12):
    out, prev = [], "<s>"
    while len(out) < max_len:
        nexts = [(b, n) for (a, b), n in COUNTS[corpus].items() if a == prev]
        prev = random.choices([b for b, _ in nexts], weights=[n for _, n in nexts])[0]
        if prev == "</s>":
            break
        out.append(prev)
    return " ".join(out)

for corpus in ("news", "nsc"):
    print(f"--- {corpus}")
    for _ in range(10):
        print("  ", sample(corpus))

**(c) A third word in the decoder.** Add "oh" = a single /oʊ/ state to `WORDS` and decode the ten clips again. Does it steal any clip? Look at where its score comes from: with no onset state, every onset frame must be paid for under /oʊ/. Then try a word with the states in the other order, "on" = /oʊ/ → /n/, and explain the result from the onset frames.

In [ ]:
for extra_name, extra_states in (("oh", ["ou"]), ("on", ["ou", "n"])):
    words3 = dict(WORDS, **{extra_name: extra_states})
    stolen = [clip for clip in sorted(LL) if decode(clip, words3)[0] == extra_name]
    print(f"with '{extra_name}' = {extra_states}: clips decoded as '{extra_name}': {stolen if stolen else 'none'}")

---
## ✅ Done looks like

- the printed ice-cream trellis matching the machine's, arrows included, and the relabelled version with nothing changed
- `viterbi` with your line in it, reproducing Part 1 in logs, and a thousand-probability product that is exactly 0.0 without them
- ten clips decoded, the accuracy in the table, one confusion explained frame by frame, and a plotted boundary you did not mark
- the same five candidates with two different winners under two priors, and the λ at which the prior takes over
- a decoder that could not say "makan" until the word was in its lexicon

**What you built.** A classical decoder in miniature: an acoustic model (given), word HMMs, Viterbi (yours), a lexicon and an n-gram prior, added in logs. Next week the acoustic model is learned from data instead of given.